## Environment setup

Clear Spark environment settings before importing and configuring PySpark.


In [1]:
import os
os.environ.pop("SPARK_HOME", None)

'/opt/spark'

## Import dependencies

Load PySpark, environment-variable support, and the Spark session builder.


In [2]:
import os
from dotenv import load_dotenv
import pyspark
from pyspark.sql import SparkSession

print(pyspark.__version__)


3.5.3


## Start the Spark session

Load environment settings, configure Delta Lake and S3 access, and verify the Spark version and UTC timezone.


In [3]:
load_dotenv()

spark_major_version = int(pyspark.__version__.split(".", 1)[0])


spark = (
    SparkSession.builder.appName("silver_transformations")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.ui.showConsoleProgress", "false")         # stops the progress bars flooding the notebook
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,io.delta:delta-spark_2.12:3.3.2")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.endpoint", "s3.eu-north-1.amazonaws.com")
    .master("local[3]")
    .getOrCreate()
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

assert spark.version == "3.5.3"
assert spark.conf.get("spark.sql.session.timeZone") == "UTC"


26/10/25 17:58:32 WARN Utils: Your hostname, zey resolves to a loopback address: 127.0.1.1; using 192.168.1.3 instead (on interface wlp9s0)
26/10/25 17:58:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/zeyad/anaconda3/envs/dataeng/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/zeyad/.ivy2/cache
The jars for the packages stored in: /home/zeyad/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-103afa18-0fd6-438b-a54c-2ae5ad8e558d;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 254ms :: artifacts dl 9ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hadoop#

## Import Spark functions

Make Spark SQL functions available for transformations and validation.


In [4]:
from pyspark.sql import functions as F

## Configure the object-storage paths

Read the bucket name and build the S3A base path used by the pipeline.


In [5]:
bucket = os.environ["S3_BUCKET"]
s3 = f"s3a://{bucket}"

## Load landing readings

Attach source-file metadata and read raw meter and transformer JSONL files from the landing area.


In [6]:
def with_file_info(df):
    return (df.withColumn("source_file", F.input_file_name())
            .withColumn("file_day", F.regexp_extract("source_file", r"_(\d{8})\.", 1))
    )

landing_meter = with_file_info(
    spark.read.text(f"{s3}/landing/meter_readings_*.jsonl")
         .withColumnRenamed("value", "raw_value"))

landing_trans = with_file_info(
    spark.read.text(f"{s3}/landing/transformer_readings_*.jsonl")
         .withColumnRenamed("value", "raw_value"))


26/10/25 17:58:42 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


## Select the relevant bronze records

Read bronze data, apply the run cutoff, and separate meter and transformer topics.


In [7]:
bronze = spark.read.parquet(f"{s3}/bronze/")

#because the data included multiple runs. This lead inconsistent anomalies and many problems , will state in document
CUTOFF = "2026-10-24 15:20:00"
bronze_real = bronze.filter(F.col("kafka_timestamp") >= CUTOFF)

bronze_meter = bronze_real.filter(F.col("topic") == "meter.readings")
bronze_trans = bronze_real.filter(F.col("topic") == "transformer.readings")

26/10/25 17:58:45 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## Combine landing and Kafka records

Normalize both sources into a shared raw schema while preserving source and Kafka metadata.


In [8]:
def to_raw(landing_df, bronze_df):
    landing = (landing_df.select("raw_value")
        .withColumn("partition", F.lit(0))
        .withColumn("offset", F.monotonically_increasing_id())
        .withColumn("kafka_timestamp", F.lit(None).cast("timestamp"))
        .withColumn("source", F.lit("landing")))
    kafka = (bronze_df.select("raw_value", "partition", "offset", "kafka_timestamp")
        .withColumn("source", F.lit("kafka")))
    return landing.unionByName(kafka)

raw_meter = to_raw(landing_meter, bronze_meter)
raw_trans = to_raw(landing_trans, bronze_trans)

## Check the filtered bronze volume

Count meter records retained after the bronze cutoff and topic filter.


In [10]:
bronze_meter.count()

1500001

## Check the combined raw volume

Count the meter records after merging landing and Kafka inputs.


In [9]:
raw_meter.count()

11149222

## Load grid reference data

Read meter and transformer reference CSVs, exclude known development leftovers, and inspect the meter schema.


In [9]:
IGNORE_GRID_DAYS = {"20260803"}   # the dev-mode leftovers

def load_grid(prefix):
    df = with_file_info(
        spark.read.option("header", True).option("encoding", "UTF-8")
             .csv(f"{s3}/grid/{prefix}_*.csv"))
    return df.filter(~F.col("file_day").isin(*IGNORE_GRID_DAYS))

grid_meters = load_grid("meters")
grid_trans = load_grid("transformers")

grid_meters.printSchema()

root
 |-- meter_id: string (nullable = true)
 |-- transformer_id: string (nullable = true)
 |-- tariff: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- lon: string (nullable = true)
 |-- source_file: string (nullable = false)
 |-- file_day: string (nullable = false)



## Load anomaly truth data

Read the simulation truth file used to compare detected anomalies with expected events.


In [10]:
truth = spark.read.json(f"{s3}/truth/sim_truth.jsonl")

truth.show(truncate=False)

+----------+--------------------+---------+----------+------+--------------+--------------------+--------------+--------------+
|anomaly_id|end_ts              |entity_id|event_type|factor|recorded_share|start_ts            |transformer_id|voltage_factor|
+----------+--------------------+---------+----------+------+--------------+--------------------+--------------+--------------+
|A-0001    |2026-08-03T00:00:00Z|M-022808 |theft     |NULL  |0.29          |2026-08-01T00:00:00Z|T-0115        |NULL          |
|A-0002    |2026-08-03T00:00:00Z|M-022861 |theft     |NULL  |0.4           |2026-08-01T00:00:00Z|T-0115        |NULL          |
|A-0003    |2026-08-03T00:00:00Z|M-022888 |theft     |NULL  |0.3           |2026-08-01T00:00:00Z|T-0115        |NULL          |
|A-0004    |2026-08-03T00:00:00Z|M-022803 |theft     |NULL  |0.32          |2026-08-01T00:00:00Z|T-0115        |NULL          |
|A-0005    |2026-08-03T00:00:00Z|M-022834 |theft     |NULL  |0.24          |2026-08-01T00:00:00Z|T-0115 

## Define the raw record schema

Import Spark data types needed to parse and cast incoming reading fields.


In [10]:
from pyspark.sql.types import StructType, StructField, StringType

## Profile missing values

Create a reusable null-profile helper to quantify missing values in raw readings.


In [13]:
def null_profile(raw, cols):
    schema = StructType([StructField(c, StringType()) for c in cols]
                        + [StructField("_corrupt_record", StringType())])
    parsed = (raw.select("source", F.from_json("raw_value", schema, opts).alias("p"))
                 .select("source", "p.*"))   # raw_value and Kafka columns are gone from here on
    blank = lambda c: F.col(c).isNull() | (F.trim(F.col(c)) == "")
    return (parsed.groupBy("source")
            .agg(F.count("*").alias("total"),
                 F.count("_corrupt_record").alias("corrupt_json"),
                 *[F.count(F.when(blank(c), 1)).alias(c) for c in cols])
            .orderBy("source"))

null_profile(raw_meter, ["meter_id", "ts", "kw", "kvar", "voltage", "kwh"]).show()
null_profile(raw_trans, ["meter_id", "ts", "kw", "kvar", "voltage", "kwh"]).show()

+-------+-------+------------+--------+----+----+----+-------+----+
| source|  total|corrupt_json|meter_id|  ts|  kw|kvar|voltage| kwh|
+-------+-------+------------+--------+----+----+----+-------+----+
|  kafka|1500001|         301|     637| 601| 301| 301|    301| 301|
|landing|9649221|        1968|    3878|3854|1968|1968|   1968|1968|
+-------+-------+------------+--------+----+----+----+-------+----+



+-------+-----+------------+--------+---+---+-----+-------+---+
| source|total|corrupt_json|meter_id| ts| kw| kvar|voltage|kwh|
+-------+-----+------------+--------+---+---+-----+-------+---+
|  kafka| 7501|           0|    7501|  0|  0| 7501|      0|  0|
|landing|48000|           0|   48000|  0|  0|48000|      0|  0|
+-------+-----+------------+--------+---+---+-----+-------+---+



## Inspect the raw meter schema

Review the inferred schema before applying parsing and quality rules.


In [14]:
raw_meter.printSchema()

root
 |-- raw_value: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- source: string (nullable = false)



## Define meter quality rules

Identify blank identifiers and other invalid records that should be flagged or rejected.


In [11]:
cols = ["meter_id", "ts", "kw", "kvar", "voltage", "kwh"]
schema = StructType([StructField(c, StringType()) for c in cols]
                    + [StructField("_corrupt_record", StringType())])
opts = {"mode": "PERMISSIVE", "columnNameOfCorruptRecord": "_corrupt_record"}

parsed = (raw_meter
    .withColumn("p", F.from_json("raw_value", schema, opts))
    .select("source", "partition", "offset", "kafka_timestamp", "raw_value", "p.*")
    .withColumn("ts_t", F.try_to_timestamp("ts")))
for c in ["kw", "kvar", "voltage", "kwh"]:
    parsed = parsed.withColumn(c + "_n", F.expr(f"try_cast({c} as double)"))

In [12]:
both = (parsed.filter("meter_id IS NOT NULL AND ts_t IS NOT NULL")
        .groupBy("meter_id", "ts_t")
        .agg(F.countDistinct("source").alias("n_src"))
        .filter("n_src > 1"))
both.count()

26/10/25 18:00:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 18:01:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 18:01:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 18:02:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 18:02:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 18:03:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 18:04:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 18:09:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 18:09:09 WARN RowBasedKeyValueBatch: Calling spill() on

21

In [13]:
d = (parsed.filter("meter_id IS NOT NULL AND ts_t IS NOT NULL")
     .join(both.select("meter_id", "ts_t"), ["meter_id", "ts_t"])
     .select("meter_id", "ts_t", "source", "kw_n", "kvar_n", "voltage_n", "kwh_n")
     .orderBy("meter_id", "ts_t", "source"))
d.show(42, False)


26/10/25 18:23:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 18:24:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 18:24:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 18:25:41 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 18:26:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 18:26:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 18:27:16 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 18:32:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 18:32:07 WARN RowBasedKeyValueBatch: Calling spill() on

+--------+-------------------+-------+-----+------+---------+------+
|meter_id|ts_t               |source |kw_n |kvar_n|voltage_n|kwh_n |
+--------+-------------------+-------+-----+------+---------+------+
|M-002022|2026-08-01 23:30:00|kafka  |2.987|1.301 |223.9    |0.7467|
|M-002022|2026-08-01 23:30:00|landing|2.987|1.301 |223.9    |0.7467|
|M-007549|2026-08-01 23:00:00|kafka  |1.048|0.459 |223.38   |0.2619|
|M-007549|2026-08-01 23:00:00|landing|1.048|0.459 |223.38   |0.2619|
|M-010116|2026-08-01 23:30:00|kafka  |2.111|0.778 |222.65   |0.5278|
|M-010116|2026-08-01 23:30:00|landing|2.111|0.778 |222.65   |0.5278|
|M-010683|2026-08-01 23:15:00|kafka  |1.433|0.661 |224.48   |0.3582|
|M-010683|2026-08-01 23:15:00|landing|1.433|0.661 |224.48   |0.3582|
|M-013996|2026-08-01 22:15:00|kafka  |2.452|0.75  |220.84   |0.613 |
|M-013996|2026-08-01 22:15:00|landing|2.452|0.75  |220.84   |0.613 |
|M-015198|2026-08-01 22:45:00|kafka  |1.821|0.761 |219.93   |0.4553|
|M-015198|2026-08-01 22:45:00|land

## Keep accepted readings

Separate records with no rejection reason from rows routed to data-quality handling.


In [29]:
blank = F.col("meter_id").isNull() | (F.trim("meter_id") == "")
num_bad = (F.col("kw_n").isNull() | F.col("kvar_n").isNull()
           | F.col("voltage_n").isNull() | F.col("kwh_n").isNull())

flagged = parsed.withColumn("reject_reason",
    F.when(F.col("_corrupt_record").isNotNull(), "malformed_json")
     .when(blank, "missing_meter_id")
     .when(~F.col("meter_id").rlike(r"^M-\d{6}$"), "bad_meter_id_format")
     .when(F.col("ts_t").isNull(), "missing_or_bad_ts")
     .when(num_bad, "non_numeric_value")
     .when(F.col("kw_n") < 0, "negative_kw"))

## Reviewing Good and Dead-letter data

In [30]:
good = flagged.filter("reject_reason IS NULL")
dlq = (flagged.filter("reject_reason IS NOT NULL")
       .select("reject_reason", "source", "partition", "offset", "kafka_timestamp", "raw_value"))

## Review the meter dead-letter data

Read previously rejected meter records for inspection and downstream reconciliation.


In [13]:
from pyspark.sql.window import Window

w = (Window.partitionBy("meter_id", "ts")
     .orderBy(F.when(F.col("source") == "landing", 0).otherwise(1), "kafka_timestamp", "partition", "offset")
    )

deduped = (good.withColumn("rn", F.row_number().over(w)).filter("rn = 1")
           .select("meter_id", F.col("ts_t").alias("ts"),
                   F.col("kw_n").alias("kw"), F.col("kvar_n").alias("kvar"),
                   F.col("voltage_n").alias("voltage"), F.col("kwh_n").alias("kwh")))

dlq.write.mode("overwrite").parquet("phase1/meter_dlq")
deduped.write.mode("overwrite").parquet("phase1/meter_clean")

In [9]:
d = spark.read.parquet("phase1/meter_dlq")
c = spark.read.parquet("phase1/meter_clean")

d.groupBy("reject_reason", "source").count().orderBy("reject_reason", "source").show()
n_dlq, n_clean = d.count(), c.count()
print("dups removed:", 11_149_222 - n_dlq - n_clean)

+-----------------+-------+-----+
|    reject_reason| source|count|
+-----------------+-------+-----+
|   malformed_json|  kafka|  301|
|   malformed_json|landing| 1968|
| missing_meter_id|  kafka|  336|
| missing_meter_id|landing| 1910|
|missing_or_bad_ts|  kafka|  300|
|missing_or_bad_ts|landing| 1886|
|      negative_kw|  kafka|  345|
|      negative_kw|landing| 1990|
|non_numeric_value|  kafka|  302|
|non_numeric_value|landing| 1924|
+-----------------+-------+-----+

dups removed: 55631


## Cleaning Transformer Readings

In [12]:
from pyspark.sql.window import Window

opts = {"mode": "PERMISSIVE", "columnNameOfCorruptRecord": "_corrupt_record"}


t_cols = ["transformer_id", "ts", "kw", "voltage", "kwh"]
t_schema = StructType([StructField(c, StringType()) for c in t_cols]
                      + [StructField("_corrupt_record", StringType())])

t = (raw_trans
    .withColumn("p", F.from_json("raw_value", t_schema, opts))
    .select("source", "partition", "offset", "kafka_timestamp", "raw_value", "p.*")
    .withColumn("ts_t", F.try_to_timestamp("ts")))
for c in ["kw", "voltage", "kwh"]:
    t = t.withColumn(c + "_n", F.expr(f"try_cast({c} as double)"))

t_flagged = t.withColumn("reject_reason",
    F.when(F.col("_corrupt_record").isNotNull(), "malformed_json")
     .when(F.col("transformer_id").isNull() | (F.trim("transformer_id") == ""), "missing_transformer_id")
     .when(~F.col("transformer_id").rlike(r"^T-\d{4}$"), "bad_transformer_id_format")
     .when(F.col("ts_t").isNull(), "missing_or_bad_ts")
     .when(F.col("kw_n").isNull() | F.col("voltage_n").isNull() | F.col("kwh_n").isNull(), "non_numeric_value")
     .when(F.col("kw_n") < 0, "negative_kw"))

t_dlq = (t_flagged.filter("reject_reason IS NOT NULL")
         .select("reject_reason", "source", "partition", "offset", "kafka_timestamp", "raw_value"))

tw = (Window.partitionBy("transformer_id", "ts_t")
      .orderBy(F.when(F.col("source") == "landing", 0).otherwise(1),
               "kafka_timestamp", "partition", "offset"))

t_clean = (t_flagged.filter("reject_reason IS NULL")
           .withColumn("rn", F.row_number().over(tw)).filter("rn = 1")
           .select("transformer_id", F.col("ts_t").alias("ts"),
                   F.col("kw_n").alias("kw"), F.col("voltage_n").alias("voltage"),
                   F.col("kwh_n").alias("kwh")))

print(t_dlq.count(), t_clean.count())

0 55500


## Build the final meter dataset


In [13]:
def grid_day(d):
    return (spark.read.option("header", True).option("encoding", "UTF-8")
            .csv(f"{s3}/grid/meters_{d}.csv")
            .select("meter_id", "transformer_id")
            .withColumn("grid_date", F.to_date(F.lit(d), "yyyyMMdd")))

IGNORE_GRID_DAYS = ["2026-08-03"]    # the dev leftovers; remove once M1 deletes them

grid = (spark.read.option("header", True).option("encoding", "UTF-8")
        .option("pathGlobFilter", "meters_*.csv")
        .csv(f"{s3}/grid/")
        .select("meter_id", "transformer_id", F.col("_metadata.file_name").alias("f"))
        .withColumn("grid_date", F.to_date(F.regexp_extract("f", r"_(\d{8})\.csv", 1), "yyyyMMdd"))
        .drop("f")
        .filter(~F.col("grid_date").cast("string").isin(*IGNORE_GRID_DAYS)))

c = spark.read.parquet("phase1/meter_clean")
joined = (c.withColumn("grid_date", F.to_date("ts"))
           .join(F.broadcast(grid), ["meter_id", "grid_date"], "left"))

n_nomatch = joined.filter("transformer_id IS NULL").count()
print("no_grid_match:", n_nomatch)          # expect 0

meter_final = (joined.filter("transformer_id IS NOT NULL")
    .select("meter_id", "transformer_id", "ts", "kw", "kvar", "voltage", "kwh")
    .withColumn("date", F.to_date("ts")).withColumn("hour", F.hour("ts")))

assert meter_final.count() == c.count() - n_nomatch   # a bigger count means the grid has duplicate meter rows

no_grid_match: 0


In [14]:
trans_final = (t_clean.withColumn("date", F.to_date("ts")).withColumn("hour", F.hour("ts")))

## Define silver output locations

Set the base directory used to write transformed and quality-checked datasets.


In [15]:
assert spark.conf.get("spark.sql.session.timeZone") == "UTC"

def write_outputs(df, name, base):
    d = df.repartition("date", "hour")
    d.write.mode("overwrite").partitionBy("date", "hour").parquet(f"{base}/export/{name}")
    d.drop("hour").write.format("delta").mode("overwrite").partitionBy("date").save(f"{base}/silver/{name}")

base = "out"                                  # local first; switch to f"{s3}" after checking
write_outputs(meter_final, "meter_readings", base)
write_outputs(trans_final, "transformer_readings", base)

26/10/25 09:30:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [16]:
base = "out"
m = spark.read.parquet(f"{base}/export/meter_readings")
t = spark.read.parquet(f"{base}/export/transformer_readings")
m.printSchema(); t.printSchema()

root
 |-- meter_id: string (nullable = true)
 |-- transformer_id: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- kw: double (nullable = true)
 |-- kvar: double (nullable = true)
 |-- voltage: double (nullable = true)
 |-- kwh: double (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)

root
 |-- transformer_id: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- kw: double (nullable = true)
 |-- voltage: double (nullable = true)
 |-- kwh: double (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)



## Inspect transformed row counts

Display the number of meter and transformer rows available for output.


In [17]:
print("Meter rows:", m.count())
print("Transformer rows:", t.count())

Meter rows: 11082329
Transformer rows: 55500


## Null Checks

In [18]:
def check_nulls(df, name):
    print(f"\n===== {name}: NULL CHECK =====")

    null_counts = df.agg(*[
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in df.columns
    ])

    null_counts.show(truncate=False)

    any_null_condition = None

    for c in df.columns:
        condition = F.col(c).isNull()
        any_null_condition = (
            condition if any_null_condition is None
            else any_null_condition | condition
        )

    print(
        "Rows containing at least one NULL:",
        df.filter(any_null_condition).count()
    )

check_nulls(m, "Meter Readings")
check_nulls(t, "Transformer Readings")


===== Meter Readings: NULL CHECK =====
+--------+--------------+---+---+----+-------+---+----+----+
|meter_id|transformer_id|ts |kw |kvar|voltage|kwh|date|hour|
+--------+--------------+---+---+----+-------+---+----+----+
|0       |0             |0  |0  |0   |0      |0  |0   |0   |
+--------+--------------+---+---+----+-------+---+----+----+

Rows containing at least one NULL: 0

===== Transformer Readings: NULL CHECK =====
+--------------+---+---+-------+---+----+----+
|transformer_id|ts |kw |voltage|kwh|date|hour|
+--------------+---+---+-------+---+----+----+
|0             |0  |0  |0      |0  |0   |0   |
+--------------+---+---+-------+---+----+----+

Rows containing at least one NULL: 0


## Duplicate Checks

In [19]:
def check_duplicates(df, name, key_columns):
    print(f"\n===== {name}: DUPLICATE CHECK =====")

    duplicated_keys = (
        df.groupBy(*key_columns)
          .count()
          .filter(F.col("count") > 1)
    )

    duplicate_key_count = duplicated_keys.count()

    # Number of extra rows beyond one row per key
    extra_duplicate_rows = (
        duplicated_keys
        .agg(F.sum(F.col("count") - 1).alias("extra_rows"))
        .first()["extra_rows"]
    ) or 0

    print("Duplicated keys:", duplicate_key_count)
    print("Extra duplicate rows:", extra_duplicate_rows)

    if duplicate_key_count > 0:
        print("Sample duplicated keys:")
        duplicated_keys.orderBy(F.desc("count")).show(20, truncate=False)

check_duplicates(m, "Meter Readings", ["meter_id", "ts"])
check_duplicates(t, "Transformer Readings", ["transformer_id", "ts"])


===== Meter Readings: DUPLICATE CHECK =====


26/10/25 09:54:16 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 09:54:16 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 09:54:16 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 09:54:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 09:54:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 09:54:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 09:54:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 09:54:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/25 09:54:18 WARN RowBasedKeyValueBatch: Calling spill() on

Duplicated keys: 0
Extra duplicate rows: 0

===== Transformer Readings: DUPLICATE CHECK =====
Duplicated keys: 0
Extra duplicate rows: 0


## Negative Value Checks

In [20]:
def check_negative_values(df, name, columns):
    print(f"\n===== {name}: NEGATIVE VALUE CHECK =====")

    available_columns = [c for c in columns if c in df.columns]

    results = df.agg(*[
        F.count(
            F.when(F.col(c) < 0, 1)
        ).alias(c)
        for c in available_columns
    ])

    results.show(truncate=False)

    for c in available_columns:
        print(f"\nSample rows where {c} is negative:")

        df.filter(F.col(c) < 0) \
          .select(
              *[
                  x for x in [
                      "meter_id",
                      "transformer_id",
                      "ts",
                      c
                  ]
                  if x in df.columns
              ]
          ) \
          .show(10, truncate=False)

# Negative values to investigate
check_negative_values(
    m,
    "Meter Readings",
    ["kw", "kvar", "voltage", "kwh"]
)

check_negative_values(
    t,
    "Transformer Readings",
    ["kw", "kvar", "voltage", "kwh"]
)


===== Meter Readings: NEGATIVE VALUE CHECK =====
+---+----+-------+---+
|kw |kvar|voltage|kwh|
+---+----+-------+---+
|0  |0   |0      |0  |
+---+----+-------+---+


Sample rows where kw is negative:
+--------+--------------+---+---+
|meter_id|transformer_id|ts |kw |
+--------+--------------+---+---+
+--------+--------------+---+---+


Sample rows where kvar is negative:
+--------+--------------+---+----+
|meter_id|transformer_id|ts |kvar|
+--------+--------------+---+----+
+--------+--------------+---+----+


Sample rows where voltage is negative:
+--------+--------------+---+-------+
|meter_id|transformer_id|ts |voltage|
+--------+--------------+---+-------+
+--------+--------------+---+-------+


Sample rows where kwh is negative:
+--------+--------------+---+---+
|meter_id|transformer_id|ts |kwh|
+--------+--------------+---+---+
+--------+--------------+---+---+


===== Transformer Readings: NEGATIVE VALUE CHECK =====
+---+-------+---+
|kw |voltage|kwh|
+---+-------+---+
|0  |0  

## Infinite Values Check

In [21]:
def check_non_finite_values(df, name, columns):
    print(f"\n===== {name}: NON-FINITE CHECK =====")

    available_columns = [c for c in columns if c in df.columns]

    checks = []

    for c in available_columns:
        value = F.col(c)

        checks.append(
            F.count(
                F.when(
                    value.isNotNull() &
                    (
                        F.isnan(value) |
                        (value == float("inf")) |
                        (value == float("-inf"))
                    ),
                    1
                )
            ).alias(c)
        )

    df.agg(*checks).show(truncate=False)

check_non_finite_values(
    m, "Meter Readings", ["kw", "kvar", "voltage", "kwh"]
)

check_non_finite_values(
    t, "Transformer Readings", ["kw", "voltage", "kwh"]
)


===== Meter Readings: NON-FINITE CHECK =====
+---+----+-------+---+
|kw |kvar|voltage|kwh|
+---+----+-------+---+
|0  |0   |0      |0  |
+---+----+-------+---+


===== Transformer Readings: NON-FINITE CHECK =====
+---+-------+---+
|kw |voltage|kwh|
+---+-------+---+
|0  |0      |0  |
+---+-------+---+



## Timestamp Assuring

In [22]:
def check_timestamp_alignment(df, name):
    print(f"\n===== {name}: TIMESTAMP ALIGNMENT =====")

    bad_timestamps = df.filter(
        F.col("ts").isNotNull() &
        (
            (~F.minute("ts").isin(0, 15, 30, 45)) |
            (F.second("ts") != 0)
        )
    )

    print("Misaligned timestamps:", bad_timestamps.count())

    bad_timestamps.select(
        *[
            c for c in ["meter_id", "transformer_id", "ts"]
            if c in df.columns
        ]
    ).show(10, truncate=False)

check_timestamp_alignment(m, "Meter Readings")
check_timestamp_alignment(t, "Transformer Readings")


===== Meter Readings: TIMESTAMP ALIGNMENT =====
Misaligned timestamps: 0
+--------+--------------+---+
|meter_id|transformer_id|ts |
+--------+--------------+---+
+--------+--------------+---+


===== Transformer Readings: TIMESTAMP ALIGNMENT =====
Misaligned timestamps: 0
+--------------+---+
|transformer_id|ts |
+--------------+---+
+--------------+---+



## Energy Consistency Check

In [23]:
def check_energy_consistency(df, name, tolerance=0.05):
    print(f"\n===== {name}: ENERGY CONSISTENCY =====")

    required = {"kw", "kwh"}

    if not required.issubset(df.columns):
        print("Required columns are missing.")
        return

    # Skip nulls and non-finite values; other checks handle those separately.
    valid = df.filter(
        F.col("kw").isNotNull() &
        F.col("kwh").isNotNull() &
        (~F.isnan("kw")) &
        (~F.isnan("kwh")) &
        (F.abs(F.col("kw")) != float("inf")) &
        (F.abs(F.col("kwh")) != float("inf"))
    )

    # Absolute tolerance in kWh.
    inconsistent = valid.filter(
        F.abs(F.col("kwh") - F.col("kw") * 0.25) > tolerance
    )

    print("Rows outside expected energy tolerance:", inconsistent.count())

    inconsistent.select(
        *[
            c for c in [
                "meter_id", "transformer_id", "ts", "kw", "kwh"
            ]
            if c in df.columns
        ]
    ).show(10, truncate=False)

check_energy_consistency(m, "Meter Readings")
check_energy_consistency(t, "Transformer Readings")


===== Meter Readings: ENERGY CONSISTENCY =====
Rows outside expected energy tolerance: 0
+--------+--------------+---+---+---+
|meter_id|transformer_id|ts |kw |kwh|
+--------+--------------+---+---+---+
+--------+--------------+---+---+---+


===== Transformer Readings: ENERGY CONSISTENCY =====
Rows outside expected energy tolerance: 0
+--------------+---+---+---+
|transformer_id|ts |kw |kwh|
+--------------+---+---+---+
+--------------+---+---+---+



In [31]:
from delta.tables import DeltaTable

S3_OUTPUT_BASE = f"s3a://{bucket}".rstrip("/")
CONFIRM_S3_OVERWRITE = True  # change to true after confirming the output path

if not CONFIRM_S3_OVERWRITE:
    print("S3 publish skipped (safe default).")
    print("Review the run-specific CUTOFF and IGNORE_GRID_DAYS filters first.")
    print(f"Target bucket: {S3_OUTPUT_BASE}")
    print(f"Delta:         {S3_OUTPUT_BASE}/silver/meter_readings")
    print(f"Delta:         {S3_OUTPUT_BASE}/silver/transformer_readings")
    print(f"Parquet:       {S3_OUTPUT_BASE}/export/meter_readings")
    print(f"Parquet:       {S3_OUTPUT_BASE}/export/transformer_readings")
    print(f"DLQ:           {S3_OUTPUT_BASE}/dlq/meter_readings")
    print(f"DLQ:           {S3_OUTPUT_BASE}/dlq/transformer_readings")
else:
    expected_meter_rows = m.count()
    expected_transformer_rows = t.count()
    expected_meter_dlq_rows = dlq.count()
    expected_transformer_dlq_rows = t_dlq.count()

    write_outputs(meter_final, "meter_readings", S3_OUTPUT_BASE)
    write_outputs(trans_final, "transformer_readings", S3_OUTPUT_BASE)

    dlq.write.mode("overwrite").parquet(f"{S3_OUTPUT_BASE}/dlq/meter_readings")
    t_dlq.write.mode("overwrite").parquet(f"{S3_OUTPUT_BASE}/dlq/transformer_readings")

    # Read-after-write verification for both plain Parquet exports and Delta Silver tables.
    meter_export_count = spark.read.parquet(f"{S3_OUTPUT_BASE}/export/meter_readings").count()
    transformer_export_count = spark.read.parquet(f"{S3_OUTPUT_BASE}/export/transformer_readings").count()
    meter_silver_path = f"{S3_OUTPUT_BASE}/silver/meter_readings"
    transformer_silver_path = f"{S3_OUTPUT_BASE}/silver/transformer_readings"
    assert DeltaTable.isDeltaTable(spark, meter_silver_path), "Meter Silver output is not a Delta table"
    assert DeltaTable.isDeltaTable(spark, transformer_silver_path), "Transformer Silver output is not a Delta table"
    meter_silver_count = spark.read.format("delta").load(meter_silver_path).count()
    transformer_silver_count = spark.read.format("delta").load(transformer_silver_path).count()
    meter_dlq_count = spark.read.parquet(f"{S3_OUTPUT_BASE}/dlq/meter_readings").count()
    transformer_dlq_count = spark.read.parquet(f"{S3_OUTPUT_BASE}/dlq/transformer_readings").count()

    assert meter_export_count == expected_meter_rows, (meter_export_count, expected_meter_rows)
    assert transformer_export_count == expected_transformer_rows, (transformer_export_count, expected_transformer_rows)
    assert meter_silver_count == expected_meter_rows, (meter_silver_count, expected_meter_rows)
    assert transformer_silver_count == expected_transformer_rows, (transformer_silver_count, expected_transformer_rows)
    assert meter_dlq_count == expected_meter_dlq_rows, (meter_dlq_count, expected_meter_dlq_rows)
    assert transformer_dlq_count == expected_transformer_dlq_rows, (transformer_dlq_count, expected_transformer_dlq_rows)

    print("S3 publish and read-after-write checks passed.")
    print(f"Meter rows: {meter_export_count:,}; transformer rows: {transformer_export_count:,}")
    print(f"Meter DLQ: {meter_dlq_count:,}; transformer DLQ: {transformer_dlq_count:,}")
    print("Delta tables verified at both silver/ paths.")


S3 publish and read-after-write checks passed.
Meter rows: 11,082,329; transformer rows: 55,500
Meter DLQ: 11,262; transformer DLQ: 0
Delta tables verified at both silver/ paths.


In [15]:
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType

STREAM_JOB_ID = "bronze_to_silver_v1"

# Choose ONE source mode based on the actual Bronze layout:
#   parquet_mixed = current notebook layout: one Parquet folder with a topic column
#   delta_mixed   = one Delta table containing a topic column
#   delta_split   = two Delta tables: meter_readings and transformer_readings
BRONZE_SOURCE_MODE = "parquet_mixed"
BRONZE_STREAM_PATH = f"{s3}/bronze"
METER_BRONZE_DELTA_PATH = f"{s3}/bronze/meter_readings"
TRANSFORMER_BRONZE_DELTA_PATH = f"{s3}/bronze/transformer_readings"

STREAM_ROOT = f"{s3}/streaming/{STREAM_JOB_ID}"
STREAM_CHECKPOINT_PATH = f"{s3}/checkpoints/{STREAM_JOB_ID}"
MAX_FILES_PER_TRIGGER = 10
STREAM_IGNORE_GRID_DAYS = {"20260803"}  # development-only grid snapshot to exclude

print("Source mode:    ", BRONZE_SOURCE_MODE)
print("Bronze source:  ", BRONZE_STREAM_PATH)
print("Checkpoint:     ", STREAM_CHECKPOINT_PATH)
print("Stream outputs: ", STREAM_ROOT)
if BRONZE_SOURCE_MODE == "parquet_mixed":
    print("Schema discovered by the existing batch reader:")
    bronze.printSchema()
    print("Columns:", bronze.columns)

    bronze.select(
        "topic",
        "raw_value",
        "partition",
        "offset",
        "kafka_timestamp"
    ).show(5, truncate=False)

Source mode:     parquet_mixed
Bronze source:   s3a://sic-grad-project/bronze
Checkpoint:      s3a://sic-grad-project/checkpoints/bronze_to_silver_v1
Stream outputs:  s3a://sic-grad-project/streaming/bronze_to_silver_v1
Schema discovered by the existing batch reader:
root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- kafka_timestamp_type: integer (nullable = true)
 |-- message_key: string (nullable = true)
 |-- raw_value: string (nullable = true)

Columns: ['topic', 'partition', 'offset', 'kafka_timestamp', 'kafka_timestamp_type', 'message_key', 'raw_value']
+--------------+---------------------------------------------------------------------------------------------------------+---------+------+-----------------------+
|topic         |raw_value                                                                                                |partition|offset|kafka_ti